# Reevaluación con 300 semillas de las redes con la recompensa corregida

Córrelo **después** de que terminen los cuadernos A y B de `mid_decision`. No entrena: evalúa las 6 redes nuevas en las mismas 300 semillas de la reevaluación anterior. Usa la misma carpeta `reeval_300`, así que el control y las heurísticas ya calculados se reutilizan y no se repiten. Dura unos **70 minutos**; si se corta, vuelve a ejecutar todo.

La última celda de reevaluación es opcional: suma las redes V4 de las semillas 3000 y 4000 que quedaron fuera la vez anterior (≈ 25 min).

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama con el parche de R_E; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-revision-opus
!cd /content/slippage && git log --oneline -1 && grep -c "mid_decision" scripts/colab/entrenamiento_inicial_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Reevaluación

In [ ]:
# Las 6 redes nuevas (opción 0: 1000-4000; opción A: 1000 y 2000), 300 semillas (≈ 70 min)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/reevaluar_ejecutores_ps.py --raices {RESULTS_DIR}/rmid_A {RESULTS_DIR}/rmid_B --n-semillas 300 --out-dir {RESULTS_DIR}/reeval_300 --max-minutes 200 2>&1 | grep -v UserWarning


In [ ]:
# OPCIONAL: redes V4 de las semillas 3000 y 4000 (≈ 25 min)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/reevaluar_ejecutores_ps.py --raices {RESULTS_DIR}/v4_pen30_g1_B --n-semillas 300 --out-dir {RESULTS_DIR}/reeval_300 --max-minutes 60 2>&1 | grep -v UserWarning


## Zip para descargar

In [ ]:
# CELDA FINAL: zip de la carpeta reeval_300 completa (solo JSON)
import os, zipfile
base = f'{RESULTS_DIR}/reeval_300'
archivos = sorted(os.path.join(base, a) for a in os.listdir(base) if a.endswith('.json'))
print(len(archivos), 'archivos')
zpath = f'{RESULTS_DIR}/reeval_300_mid.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print('zip listo:', zpath)
